In [1]:
import zipfile
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
import io
import os

In [2]:
MAIN_ZIP = "F:/daneshkar/tennis project/tennis_data.zip"

OUT_DIR = "F:/daneshkar/tennis project/tennis_outputs_csv_final"

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
df_names = [
    "votes", "power", "statistics", "pbp", "odds",
    "venue", "tournament", "time", "season", "round",
     "event", "home_team", "home_team_score" , "away_team", "away_team_score", 
]
inner_base = {
    "votes": "data/raw/raw_votes_parquet",
    "power": "data/raw/raw_tennis_power_parquet",
    "statistics": "data/raw/raw_statistics_parquet",
    "pbp": "data/raw/raw_point_by_point_parquet",
    "odds": "data/raw/raw_odds_parquet",

    "venue": "data/raw/raw_match_parquet",
    "tournament": "data/raw/raw_match_parquet",
    "time": "data/raw/raw_match_parquet",
    "season": "data/raw/raw_match_parquet",
    "round": "data/raw/raw_match_parquet",
    "event": "data/raw/raw_match_parquet",
    "home_team": "data/raw/raw_match_parquet",
    "home_team_score": "data/raw/raw_match_parquet",
    "away_team": "data/raw/raw_match_parquet",
    "away_team_score": "data/raw/raw_match_parquet",
}
    


In [4]:
with zipfile.ZipFile(MAIN_ZIP, 'r') as main_zip:
    internal_zips = [name for name in main_zip.namelist() if name.endswith(".zip")]

    print(f"Found {len(internal_zips)} internal ZIPs")

    for df_name in df_names:
        print(f"\nProcessing {df_name}")

        dfs = []
        prefix = f"{df_name}_"
        target_folder = inner_base[df_name]

        for inner_zip_name in internal_zips:
            # خواندن بایت‌های ZIP داخلی
            inner_zip = main_zip.read(inner_zip_name)
            with zipfile.ZipFile(io.BytesIO(inner_zip)) as inner_zip:

                for member in inner_zip.namelist():

                    if not member.endswith(".parquet"):
                        continue

                    if not os.path.basename(member).startswith(prefix):
                        continue

                    if target_folder not in member:
                        continue

                    parquet_file = inner_zip.read(member)
                    table = pq.read_table(pa.BufferReader(parquet_file))
                    dfs.append(table.to_pandas())

        if not dfs:
            print("WARNING: no parquet found for", df_name)
            continue

        df = pd.concat(dfs, ignore_index=True)

        out_csv = os.path.join(OUT_DIR, f"{df_name}.csv")
        df.to_csv(out_csv, index=False, encoding="utf-8")
        print("Saved:", out_csv, "rows:", len(df))


Found 60 internal ZIPs

Processing votes
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\votes.csv rows: 35658

Processing power
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\power.csv rows: 469677

Processing statistics
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\statistics.csv rows: 1358234

Processing pbp
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\pbp.csv rows: 2549369

Processing odds
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\odds.csv rows: 60946

Processing venue
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\venue.csv rows: 35423

Processing tournament
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\tournament.csv rows: 35671

Processing time
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\time.csv rows: 35671

Processing season
Saved: F:/daneshkar/tennis project/tennis_outputs_csv_final\season.csv rows: 35671

Processing round
Saved: F:/daneshkar/tennis project/tenni

In [7]:
out_path1 = 'F:/daneshkar/tennis project/tennis_outputs_csv_final/home_team_final.csv'
columns_to_drop_from_home_team = [
    'current_score', 'display_score', 'period_1', 'period_2', 'period_3',
    'period_4', 'period_5', 'period_1_tie_break', 'period_2_tie_break',
    'period_3_tie_break', 'period_4_tie_break', 'period_5_tie_break', 'normal_time'
]

home_team_df = pd.read_csv('F:/daneshkar/tennis project/tennis_outputs_csv_final/home_team.csv')
home_team_df.drop(columns=columns_to_drop_from_home_team, errors='ignore').to_csv(out_path1, index=False)

In [8]:
out_path2 = 'F:/daneshkar/tennis project/tennis_outputs_csv_final/away_team_final.csv'
columns_to_drop_from_away_team = [
    'current_score', 'display_score', 'period_1', 'period_2', 'period_3',
    'period_4', 'period_5', 'period_1_tie_break', 'period_2_tie_break',
    'period_3_tie_break', 'period_4_tie_break', 'period_5_tie_break', 'normal_time'
]

away_team_df = pd.read_csv('F:/daneshkar/tennis project/tennis_outputs_csv_final/away_team.csv')
away_team_df.drop(columns=columns_to_drop_from_away_team, errors='ignore').to_csv(out_path2, index=False)